# Import Needed Package


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = pd.read_csv("/content/drive/MyDrive/LabAI/IMDB Dataset.csv")

In [4]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
data.shape

(50000, 2)

# Data Encode

In [6]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

# Data Preprocessing

In [7]:
!pip install tensorflow

In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [10]:
train_data.shape

(40000, 2)

In [11]:
test_data.shape

(10000, 2)

In [12]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['review'])

#tokenizer untuk bikin kamus kata, untuk tentukan jumlahnya berarti akan cari 5000 kata dalam data yang akan dimasukkan kedalam kamus.

In [13]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [14]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [15]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

In [16]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


# Model Building

In [17]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid')) #untuk keluarin prediksi

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [20]:
model.fit(X_train, Y_train, batch_size=64, epochs=5, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 218s 414ms/step - accuracy: 0.7251 - loss: 0.5348 - val_accuracy: 0.8385 - val_loss: 0.4000
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 206s 412ms/step - accuracy: 0.8588 - loss: 0.3487 - val_accuracy: 0.8486 - val_loss: 0.3675
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 224s 448ms/step - accuracy: 0.8224 - loss: 0.4042 - val_accuracy: 0.7869 - val_loss: 0.4540
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 208s 416ms/step - accuracy: 0.8620 - loss: 0.3346 - val_accuracy: 0.8661 - val_loss: 0.3254
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 213s 427ms/step - accuracy: 0.8995 - loss: 0.2537 - val_accuracy: 0.8769 - val_loss: 0.3029


In [21]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 121ms/step - accuracy: 0.8826 - loss: 0.2866


In [22]:
print(loss)

0.2843133807182312


In [23]:
print(accuracy)

0.8849999904632568


Building Predictive System

In [25]:
def predictive_system(review):
  sequences = tokenizer.texts_to_sequences([review])
  padded_sequences = pad_sequences(sequences, maxlen=200)
  prediction = model.predict(padded_sequences)
  sentiment = "positive" if prediction [0][0] > 0.5 else "negative"
  return sentiment

In [26]:
predictive_system("This movie was fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 711ms/step


'positive'

In [27]:
predictive_system("A trilling adventure with stunning visual")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step


'positive'

In [28]:
predictive_system("A visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


'positive'

In [33]:
predictive_system("Overall long and slow")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step


'negative'

Saving Model

In [34]:
model.save("model.h5")

In [35]:
import joblib
joblib.dump(tokenizer,"tokenizer.pkl")

['tokenizer.pkl']